# Retrieval experiments — hybrid RRF and embedding A/B (issue #11)

Reruns the full evaluation harness (`run_eval.py`: both LangSmith datasets, all
ragas metrics, judge `openai/gpt-5-mini`) against four pipeline variants and
produces one combined results table:

| variant | retriever | embeddings |
|---|---|---|
| baseline | dense top-5 | text-embedding-3-small |
| hybrid RRF | BM25 + dense, reciprocal rank fusion (k=60) | text-embedding-3-small |
| 3-large | dense top-5 | text-embedding-3-large |
| hybrid + 3-large | BM25 + dense RRF | text-embedding-3-large |

Hybrid was chosen because Mustang jargon ("S550", "5.0", "Coyote") is
exact-match heavy. Each non-baseline variant runs 51 synthetic + 10 golden
samples (~10-25 min); finished runs are cached in `results/<variant>.json` —
delete a file to rerun that variant. Baseline aggregates are loaded from the
committed `results/baseline.json`, never rerun.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

import run_eval
from rag_pipeline import ARTIFACT, VECTOR_FILES, gateway_embeddings

RESULTS = Path("results")

# Pipeline configs (RagPipeline kwargs) per variant; keys double as LangSmith
# experiment prefixes.
VARIANTS = {
    "baseline-dense": {},
    "hybrid-rrf": {"retriever": "hybrid"},
    "dense-3large": {"embedding_model": "openai/text-embedding-3-large"},
    "hybrid-3large": {"retriever": "hybrid", "embedding_model": "openai/text-embedding-3-large"},
}
RESULT_FILES = {"baseline-dense": "baseline"}  # baseline.json is the committed name

/Users/brendanturpin/github_repos/ai-engineer-certification-challenge/evals/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3-large index

Re-embed all 993 chunks with `openai/text-embedding-3-large` via the gateway
(batches of 100) into `data/vectors_3large.npz` — float32, unit-norm, same row
order as `chunks.jsonl`. Skipped when the file already exists (it's committed).

In [2]:
vec3l = VECTOR_FILES["openai/text-embedding-3-large"]
if not vec3l.exists():
    chunks = [json.loads(line) for line in open(ARTIFACT / "chunks.jsonl")]
    emb = gateway_embeddings("openai/text-embedding-3-large")
    raw = await emb.aembed_documents([c["text"] for c in chunks], chunk_size=100)
    vectors = np.asarray(raw, dtype=np.float32)
    vectors /= np.linalg.norm(vectors, axis=1, keepdims=True)
    np.savez_compressed(vec3l, vectors=vectors)
vectors = np.load(vec3l)["vectors"]
print(vec3l, vectors.shape, vectors.dtype, "unit-norm ok:", np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-3))

/Users/brendanturpin/github_repos/ai-engineer-certification-challenge/evals/data/vectors_3large.npz (993, 3072) float32 unit-norm ok: True


## Run the variants

`results_for` reruns the full harness for any variant via `run_eval.run`
(LangSmith experiments named `<variant>-synthetic-…` / `<variant>-golden-…`),
or loads the cached aggregates if `results/<variant>.json` exists.

In [3]:
async def results_for(variant: str) -> dict:
    name = RESULT_FILES.get(variant, variant)
    path = RESULTS / f"{name}.json"
    if path.exists():
        print(f"{variant}: loaded cached {path}")
        return json.loads(path.read_text())
    return await run_eval.run(variant, pipeline_config=VARIANTS[variant], out_name=name)

reports = {"baseline-dense": await results_for("baseline-dense")}

baseline-dense: loaded cached results/baseline.json


In [4]:
reports["hybrid-rrf"] = await results_for("hybrid-rrf")

hybrid-rrf: loaded cached results/hybrid-rrf.json


In [5]:
reports["dense-3large"] = await results_for("dense-3large")

dense-3large: loaded cached results/dense-3large.json


In [6]:
reports["hybrid-3large"] = await results_for("hybrid-3large")

hybrid-3large: loaded cached results/hybrid-3large.json


In [8]:
reports

{'baseline-dense': {'judge': 'openai/gpt-5-mini',
  'config': {'retriever': 'dense', 'top_k': 5},
  'synthetic': {'experiment': 'baseline-dense-synthetic-1692f615',
   'url': 'https://smith.langchain.com/o/bf16fd47-1b69-572d-9217-aa83daeeb380/projects/p/aa4ecddc-267a-4cac-a563-e6bee71ae069',
   'samples': 51,
   'metrics': {'faithfulness': {'mean': 0.9187, 'scored': 51, 'failed': 0},
    'answer_relevancy': {'mean': 0.6358, 'scored': 51, 'failed': 0},
    'context_precision': {'mean': 0.7564, 'scored': 51, 'failed': 0},
    'context_recall': {'mean': 0.6638, 'scored': 51, 'failed': 0}}},
  'golden': {'experiment': 'baseline-dense-golden-ff0f756c',
   'url': 'https://smith.langchain.com/o/bf16fd47-1b69-572d-9217-aa83daeeb380/projects/p/0756e668-1365-41a3-90ca-37e02cec5c64',
   'samples': 10,
   'metrics': {'answer_correctness': {'mean': 0.6473,
     'scored': 10,
     'failed': 0}}}},
 'hybrid-rrf': {'judge': 'openai/gpt-5-mini',
  'config': {'retriever': 'hybrid',
   'top_k': 5,
   'em

## Combined results

In [7]:
LABELS = {
    "baseline-dense": "baseline (dense, 3-small)",
    "hybrid-rrf": "hybrid RRF (3-small)",
    "dense-3large": "dense (3-large)",
    "hybrid-3large": "hybrid RRF (3-large)",
}
rows = {}
for variant, rep in reports.items():
    row = {m: v["mean"] for m, v in rep["synthetic"]["metrics"].items()}
    row |= {m: v["mean"] for m, v in rep["golden"]["metrics"].items()}
    rows[LABELS[variant]] = row
    print(f"{variant:15s} experiments: {rep['synthetic']['experiment']}  /  {rep['golden']['experiment']}")

df = pd.DataFrame(rows).T
(RESULTS / "comparison.md").write_text(df.to_markdown(floatfmt=".4f") + "\n")
df.round(4)

baseline-dense  experiments: baseline-dense-synthetic-1692f615  /  baseline-dense-golden-ff0f756c
hybrid-rrf      experiments: hybrid-rrf-synthetic-4a1069f1  /  hybrid-rrf-golden-9ac49553
dense-3large    experiments: dense-3large-synthetic-974567f4  /  dense-3large-golden-f5185976
hybrid-3large   experiments: hybrid-3large-synthetic-f949d089  /  hybrid-3large-golden-8921816a


,faithfulness,answer_relevancy,context_precision,context_recall,answer_correctness
"baseline (dense, 3-small)",0.9187,0.6358,0.7564,0.6638,0.6473
hybrid RRF (3-small),0.9301,0.6560,0.7907,0.7426,0.6422
dense (3-large),0.9158,0.5897,0.7645,0.6427,0.6398
hybrid RRF (3-large),0.9278,0.6508,0.7955,0.6893,0.6316


## Readout

**Winner: hybrid RRF with the existing 3-small embeddings.** It takes or ties
the best score on every synthetic metric — the big move is retrieval quality,
exactly where BM25 was expected to help on exact-match Mustang jargon:
context_recall **0.6638 → 0.7426** (+0.079) and context_precision
**0.7564 → 0.7907** (+0.034) over the dense baseline, with faithfulness
(0.9187 → 0.9301) and answer_relevancy (0.6358 → 0.6560) also up.

The embedding A/B is a wash-to-negative: dense 3-large loses to the 3-small
baseline on recall (0.6427 vs 0.6638) and answer_relevancy (0.5897 vs 0.6358),
so the larger model buys nothing on this corpus. Combining it with hybrid
(hybrid-3large) edges out the best context_precision (0.7955) but gives back
recall vs hybrid-3-small (0.6893 vs 0.7426).

answer_correctness on the 10-stub golden set is flat across all four
(0.63–0.65)

**Recommendation:** ship hybrid BM25+dense RRF, keep text-embedding-3-small.